In [ ]:
import pandas as pd
import numpy as np
from hierarchicalforecast.utils import aggregate

In [ ]:

# ---------- Load Y_product ----------
Y_product = pd.read_csv("../eabl parsed csv/Y_product.csv")

# CSV loses the date type — convert it back before doing anything else
Y_product["ds"] = pd.to_datetime(Y_product["ds"])

# ---------- Load S_product ----------
# index_col=0 tells pandas the first column (unique_id) is the row index,
# exactly how it was structured before saving
S_product = pd.read_csv("../eabl parsed csv/S_product.csv", index_col=0)

In [ ]:
# ---------- Basic checks before trusting the data ----------
print("Y_product shape:", Y_product.shape)
print("Y_product columns:", Y_product.columns.tolist())
print("Distinct series in Y_product:", Y_product["unique_id"].nunique())
print("Date range:", Y_product["ds"].min(), "to", Y_product["ds"].max())
print()
print("S_product shape:", S_product.shape)
print(Y_product.head())


In [ ]:

# ---------- Check for missing values in y ----------
print("Missing values in y:", Y_product["y"].isna().sum())

# ---------- Check for duplicate (unique_id, ds) rows ----------
duplicates = Y_product.duplicated(subset=["unique_id", "ds"]).sum()
print("Duplicate unique_id/ds rows:", duplicates)

# ---------- Sort properly ----------
Y_product = Y_product.sort_values(["unique_id", "ds"]).reset_index(drop=True)

# ---------- Cross-check Y_product against S_product ----------
y_ids = set(Y_product["unique_id"].unique())
s_ids = set(S_product.index)
print("Series in Y_product but not in S_product:", len(y_ids - s_ids))
print("Series in S_product but not in Y_product:", len(s_ids - y_ids))


In [ ]:
print(S_product.index[:10])
print(S_product.index.dtype)


In [ ]:
raw = pd.read_csv("../eabl parsed csv/S_product.csv")
print(raw.columns[:10].tolist())




In [ ]:
print(raw.head())

In [ ]:
S_product = pd.read_csv("../eabl parsed csv/S_product.csv", index_col="unique_id")
S_product = S_product.drop(columns=["Unnamed: 0"], errors="ignore")

print(S_product.shape)
print(S_product.index[:10].tolist())

y_ids = set(Y_product["unique_id"].unique())
s_ids = set(S_product.index)
print("Series in Y_product but not in S_product:", len(y_ids - s_ids))
print("Series in S_product but not in Y_product:", len(s_ids - y_ids))


In [ ]:


# ---------- Get the list of bottom-level SKUs directly from S_product ----------
bottom_level_ids = S_product.columns.tolist()
print("Number of bottom-level SKUs:", len(bottom_level_ids))

# ---------- Build a full weekly calendar matching Y_product's date range ----------
full_calendar = pd.date_range(Y_product["ds"].min(), Y_product["ds"].max(), freq="W-MON")

# ---------- The classification rule (Syntetos & Boylan, 2005) ----------
def classify_demand(series):
    nonzero = series.dropna()
    nonzero = nonzero[nonzero > 0]
    if len(nonzero) == 0:
        return "no_demand"
    adi = len(series) / len(nonzero)                    # average gap between sales
    cv2 = (nonzero.std() / nonzero.mean()) ** 2          # how much sale size varies
    if adi < 1.32 and cv2 < 0.49:
        return "smooth"
    elif adi >= 1.32 and cv2 < 0.49:
        return "intermittent"
    elif adi < 1.32 and cv2 >= 0.49:
        return "erratic"
    else:
        return "lumpy"

# ---------- Classify every bottom-level SKU ----------
results = []
for uid in bottom_level_ids:
    series = (
        Y_product[Y_product["unique_id"] == uid]
        .set_index("ds")["y"]
        .reindex(full_calendar)
    )
    results.append({"unique_id": uid, "class": classify_demand(series)})

classification_product = pd.DataFrame(results)

# ---------- Check the results ----------
print(classification_product["class"].value_counts())
print(classification_product.head())

# ---------- Save it, so you don't have to rebuild this again next time ----------
classification_product.to_csv("classification_product.csv", index=False)


In [ ]:
# ---------- Get the SKUs that were directly classified in Step 2 ----------
standard_classified = classification_product[
    classification_product["class"].isin(["smooth", "erratic"])
]["unique_id"].tolist()

sparse_classified = classification_product[
    classification_product["class"].isin(["intermittent", "lumpy"])
]["unique_id"].tolist()

# ---------- Identify the higher-level rollup series (never individually classified) ----------
all_ids = Y_product["unique_id"].unique().tolist()
bottom_level_ids_set = set(classification_product["unique_id"])
rollup_ids = [uid for uid in all_ids if uid not in bottom_level_ids_set]

# ---------- Build the final two groups ----------
standard_ids = standard_classified + rollup_ids   # smooth/erratic SKUs + all rollup levels
sparse_ids = sparse_classified                     # only intermittent/lumpy SKUs

standard_df = Y_product[Y_product["unique_id"].isin(standard_ids)]
sparse_df = Y_product[Y_product["unique_id"].isin(sparse_ids)]

# ---------- Check the split ----------
print("Standard group — series count:", standard_df["unique_id"].nunique())
print("Sparse group — series count:", sparse_df["unique_id"].nunique())
print("Total (should equal 319):", standard_df["unique_id"].nunique() + sparse_df["unique_id"].nunique())
print()
print("Rollup-level series defaulted into standard group:", len(rollup_ids))
